# ДЗ 4: Olist у PostgreSQL

Працюю з датасетом Olist Brazilian E-Commerce. Спочатку закидаю шість CSV у raw-таблиці, потім переношу їх у typed-схему з ключами й обмеженнями. Далі йдуть DML-запити, сегментація покупців, JOIN-и, агрегати, set operation і window function. Postgres піднімаю через `pgserver`, CSV качаю через `kagglehub`.

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub sqlparse

import os
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw4_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri().replace("postgresql://", "postgresql+psycopg2://"), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 106.3 MB/s eta 0:00:00
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [2]:
# Варіант A: якщо Kaggle credentials налаштовані
import kagglehub

dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print(dataset_dir)

# Варіант B: якщо CSV вже лежать у репозиторії або course archive
# dataset_dir = '/content/goit-rdb-hw-04/data/olist'


Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
/kaggle/input/brazilian-ecommerce


## 1. Дані й typed-схема

Raw-таблиці — це просто CSV як є, без типів і ключів. Усі наступні запити працюють тільки з typed-таблицями, де вже є PK, FK, `CHECK` і `NOT NULL`. В оригінальному CSV колонки називаються `product_name_lenght` і `product_description_lenght` (з помилкою), тому в `SELECT` з raw беру саме ці назви, а в typed-схемі перейменовую їх на `..._length`.

In [3]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
    'olist_order_reviews_raw': 'olist_order_reviews_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    print(table_name, df.shape)


olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)
olist_order_reviews_raw (99224, 7)


In [4]:
from sqlalchemy.exc import SQLAlchemyError
from IPython.display import display
import sqlparse

def execute_sql(
    sql,
    params=None,
    return_result=False,
    output_commands=True,
):
    """
    Execute a SQL script containing one or many SQL statements.

    Every statement is executed in order using the SAME database connection.

    Output:
    - SELECT / ... RETURNING -> displays a pandas DataFrame
    - INSERT / UPDATE / DELETE -> prints affected row count
    - CREATE / DROP / ALTER, etc. -> prints success
    - If output_commands=True, prints each SQL statement before execution
    - All statements are committed together if the whole script succeeds
    - On error, the transaction is rolled back

    Parameters
    ----------
    sql : str
        One SQL statement or a script containing multiple statements.

    params : dict, optional
        Named SQLAlchemy parameters, e.g. {"id": 1} for ":id".

    return_result : bool, default=False
        If True, returns a list with execution results.

    output_commands : bool, default=True
        If True, prints each SQL statement before executing it.

    Returns
    -------
    list[dict] | None
        One result dictionary per executed SQL statement if
        return_result=True, otherwise None.
    """

    statements = [
        statement.strip()
        for statement in sqlparse.split(sql)
        if statement.strip()
    ]

    outputs = []

    try:
        with engine.begin() as conn:
            for i, statement in enumerate(statements, start=1):
                print(f"\n{'=' * 70}")
                print(f"Statement {i}/{len(statements)}")
                print(f"{'=' * 70}")

                if output_commands:
                    print(statement)
                    print("-" * 70)

                result = conn.execute(
                    text(statement),
                    params or {}
                )

                if result.returns_rows:
                    rows = result.fetchall()

                    df = pd.DataFrame(
                        rows,
                        columns=result.keys()
                    )

                    display(df)

                    outputs.append({
                        "statement_number": i,
                        "statement": statement,
                        "type": "result_set",
                        "data": df,
                    })

                else:
                    rowcount = result.rowcount

                    if rowcount is not None and rowcount >= 0:
                        print(
                            "Command executed successfully. "
                            f"Rows affected: {rowcount}"
                        )
                    else:
                        print("Command executed successfully.")

                    outputs.append({
                        "statement_number": i,
                        "statement": statement,
                        "type": "command",
                        "rowcount": rowcount,
                    })

        print(
            f"\nFinished successfully: "
            f"{len(statements)} statement(s) executed."
        )

        if return_result:
            return outputs

        return None

    except SQLAlchemyError as e:
        print(f"\nSQL error while executing statement {i}:")
        print(statement)
        print(f"\n{e}")
        raise

In [5]:
execute_sql("""
DROP TABLE IF EXISTS customer_segments    CASCADE;
DROP TABLE IF EXISTS seller_score         CASCADE;
DROP TABLE IF EXISTS seller_alerts        CASCADE;
DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
DROP TABLE IF EXISTS olist_order_items    CASCADE;
DROP TABLE IF EXISTS olist_orders         CASCADE;
DROP TABLE IF EXISTS olist_customers      CASCADE;
DROP TABLE IF EXISTS olist_products       CASCADE;
DROP TABLE IF EXISTS olist_sellers        CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE INDEX idx_olist_customers_unique_id
    ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE INDEX idx_olist_order_items_seller_order
    ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id                TEXT,
    order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title     TEXT,
    review_comment_message   TEXT,
    review_creation_date     TIMESTAMP,
    review_answer_timestamp  TIMESTAMP
);

CREATE INDEX idx_olist_order_reviews_order_id
    ON olist_order_reviews (order_id);

""")


Statement 1/19
DROP TABLE IF EXISTS customer_segments    CASCADE;
----------------------------------------------------------------------
Command executed successfully.

Statement 2/19
DROP TABLE IF EXISTS seller_score         CASCADE;
----------------------------------------------------------------------
Command executed successfully.

Statement 3/19
DROP TABLE IF EXISTS seller_alerts        CASCADE;
----------------------------------------------------------------------
Command executed successfully.

Statement 4/19
DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
----------------------------------------------------------------------
Command executed successfully.

Statement 5/19
DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
----------------------------------------------------------------------
Command executed successfully.

Statement 6/19
DROP TABLE IF EXISTS olist_order_items    CASCADE;
----------------------------------------------------------------------
Command executed 

In [6]:
execute_sql("""
CREATE INDEX IF NOT EXISTS idx_orders_customer_id
ON olist_orders(customer_id);

CREATE INDEX IF NOT EXISTS idx_order_items_order_id
ON olist_order_items(order_id);

CREATE INDEX IF NOT EXISTS idx_order_items_seller_id
ON olist_order_items(seller_id);
""")


Statement 1/3
CREATE INDEX IF NOT EXISTS idx_orders_customer_id
ON olist_orders(customer_id);
----------------------------------------------------------------------
Command executed successfully.

Statement 2/3
CREATE INDEX IF NOT EXISTS idx_order_items_order_id
ON olist_order_items(order_id);
----------------------------------------------------------------------
Command executed successfully.

Statement 3/3
CREATE INDEX IF NOT EXISTS idx_order_items_seller_id
ON olist_order_items(seller_id);
----------------------------------------------------------------------
Command executed successfully.

Finished successfully: 3 statement(s) executed.


In [7]:
execute_sql("""
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
SELECT
    review_id,
    order_id,
    review_score::SMALLINT,
    review_comment_title,
    review_comment_message,
    review_creation_date::TIMESTAMP,
    review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;
""")


Statement 1/6
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;
----------------------------------------------------------------------
Command executed successfully. Rows affected: 99441

Statement 2/6
INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;
----------------------------------------------------------------------
Command executed successfully. Rows affected: 99441

Statement 3/6
INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g:

In [8]:
execute_sql("""
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
UNION ALL SELECT 'reviews', COUNT(*) FROM olist_order_reviews
ORDER BY table_name;
""")


Statement 1/1
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
UNION ALL SELECT 'reviews', COUNT(*) FROM olist_order_reviews
ORDER BY table_name;
----------------------------------------------------------------------


,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,reviews,99224
5,sellers,3095



Finished successfully: 1 statement(s) executed.


**Smoke test.** У typed-таблицях стільки ж рядків, скільки в raw (див. вивід завантаження CSV вище): customers і orders по 99 441, order_items 112 650, products 32 951, sellers 3 095, reviews 99 224. Тобто під час перенесення нічого не загубилося.

In [9]:
execute_sql("""SELECT * FROM olist_customers LIMIT 5;""")


Statement 1/1
SELECT * FROM olist_customers LIMIT 5;
----------------------------------------------------------------------


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



Finished successfully: 1 statement(s) executed.


## 2. DML

Тут чотири запити: `INSERT ... RETURNING`, `INSERT ... ON CONFLICT`, `UPDATE ... FROM` і `DELETE ... RETURNING`.

In [10]:
execute_sql("""
DROP TABLE IF EXISTS seller_alerts CASCADE;

CREATE TABLE seller_alerts (
    alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
    severity   SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details    TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT
        so.seller_id,
        COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o
        ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO seller_alerts (seller_id, alert_type, severity, details)
SELECT
    seller_id,
    'late_delivery',
    4,
    'Seller has ' || late_orders || ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING alert_id, seller_id, alert_type, severity, created_at;
""")


Statement 1/3
DROP TABLE IF EXISTS seller_alerts CASCADE;
----------------------------------------------------------------------
Command executed successfully.

Statement 2/3
CREATE TABLE seller_alerts (
    alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
    severity   SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details    TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
----------------------------------------------------------------------
Command executed successfully.

Statement 3/3
WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT
        so.seller_id,
        COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o
        ON o.order_id = so.o

,alert_id,seller_id,alert_type,severity,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,2026-10-01 21:53:42.971049+00:00



Finished successfully: 3 statement(s) executed.


**Що робить запит.** Шукаю продавця, у якого найбільше замовлень доставлено пізніше обіцяної дати, і створюю для нього alert. Завдяки `RETURNING` одразу видно, який рядок з’явився (`alert_id`, `seller_id`, час створення). Цей вивід і працює як audit-log, окремий `SELECT` не потрібен.

Щодо повторного запуску: при Run All таблиця спершу дропається й створюється заново, тому alert завжди один. Якщо ж перезапустити тільки `INSERT` без `DROP`, з’явиться другий такий самий alert.

In [11]:
execute_sql("""
DROP TABLE IF EXISTS seller_score CASCADE;

CREATE TABLE seller_score (
    seller_id   TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating  NUMERIC(3, 2),
    n_reviews   INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier        TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
    updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT
        so.seller_id,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
        COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
SELECT
    seller_id,
    avg_rating,
    n_reviews,
    CASE
        WHEN avg_rating IS NULL THEN NULL
        WHEN avg_rating >= 4.5 THEN 'gold'
        WHEN avg_rating >= 3.5 THEN 'silver'
        ELSE 'bronze'
    END AS tier
FROM seller_rating
ON CONFLICT (seller_id) DO UPDATE
SET avg_rating = EXCLUDED.avg_rating,
    n_reviews  = EXCLUDED.n_reviews,
    tier       = EXCLUDED.tier,
    updated_at = NOW()
RETURNING seller_id, avg_rating, n_reviews, tier, updated_at;
""")


Statement 1/3
DROP TABLE IF EXISTS seller_score CASCADE;
----------------------------------------------------------------------
Command executed successfully.

Statement 2/3
CREATE TABLE seller_score (
    seller_id   TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating  NUMERIC(3, 2),
    n_reviews   INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier        TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
    updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
----------------------------------------------------------------------
Command executed successfully.

Statement 3/3
WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT
        so.seller_

,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-10-01 21:53:43.172795+00:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-10-01 21:53:43.172795+00:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-10-01 21:53:43.172795+00:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-10-01 21:53:43.172795+00:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-10-01 21:53:43.172795+00:00
...,...,...,...,...,...
3090,b2eecf5ea250510da76590ca79d60e5d,5.00,9,gold,2026-10-01 21:53:43.172795+00:00
3091,eae9af4811c294c56795d70e715b7337,5.00,1,gold,2026-10-01 21:53:43.172795+00:00
3092,1c83418358f249c9d6affa52d579db0e,5.00,1,gold,2026-10-01 21:53:43.172795+00:00
3093,845e8036c3f7bf62099352f5ed910240,3.95,21,silver,2026-10-01 21:53:43.172795+00:00



Finished successfully: 3 statement(s) executed.


**Пояснення до `seller_score`.**
- `SELECT DISTINCT` у `seller_orders` залишає одну пару `seller_id × order_id`, навіть якщо в продавця кілька позицій в одному замовленні.
- Відгуки окремо зводжу до рівня `order_id` (`review_per_order`). Інакше при join з `order_items` одна оцінка рахувалася б стільки разів, скільки позицій у замовленні.
- `ON CONFLICT (seller_id) DO UPDATE` при повторному запуску просто перезаписує метрики вже наявного продавця, нових рядків не додається.
- Якщо в продавця немає жодного відгуку, `LEFT JOIN` дає `NULL`: `avg_rating` і `tier` лишаються `NULL`, а `COUNT` повертає `n_reviews = 0`.

Обмеження: в Olist відгук прив’язаний до замовлення, а не до продавця. Якщо в замовленні товари від кількох продавців, та сама оцінка дістається кожному з них.

Нижче кілька перевірочних запитів. Видно, що одне замовлення може мати до 21 позиції, а відгук стосується замовлення загалом.

In [12]:
execute_sql("""SELECT order_id, COUNT(*) FROM olist_order_items GROUP BY order_id ORDER BY COUNT(*) DESC LIMIT 5;""")


Statement 1/1
SELECT order_id, COUNT(*) FROM olist_order_items GROUP BY order_id ORDER BY COUNT(*) DESC LIMIT 5;
----------------------------------------------------------------------


,order_id,count
0,8272b63d03f5f79c56e9e4120aec44ef,21
1,1b15974a0141d54e36626dca3fdc731a,20
2,ab14fdcfbe524636d65ee38360e22ce8,20
3,428a2f660dc84138d969ccd69a0ab6d5,15
4,9ef13efd6949e4573a18964dd1bbe7f5,15



Finished successfully: 1 statement(s) executed.


In [13]:
execute_sql("""SELECT * FROM olist_order_items LIMIT 5;""")


Statement 1/1
SELECT * FROM olist_order_items LIMIT 5;
----------------------------------------------------------------------


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14



Finished successfully: 1 statement(s) executed.


In [14]:
execute_sql("""
SELECT * FROM olist_order_reviews LIMIT 5;
""")


Statement 1/1
SELECT * FROM olist_order_reviews LIMIT 5;
----------------------------------------------------------------------


,review_row_id,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,1,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18,2018-01-18 21:46:59
1,2,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10,2018-03-11 03:05:13
2,3,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17,2018-02-18 14:36:24
3,4,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,None,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,5,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,None,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53



Finished successfully: 1 statement(s) executed.


In [15]:
execute_sql("""
ALTER TABLE olist_orders
    ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;

WITH delivery_flags AS (
    SELECT
        order_id,
        (
            order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE
        ) AS is_late_calc
    FROM olist_orders
)
UPDATE olist_orders AS o
SET is_late = f.is_late_calc
FROM delivery_flags AS f
WHERE f.order_id = o.order_id
RETURNING o.order_id, o.customer_id, o.is_late;

""")


Statement 1/2
ALTER TABLE olist_orders
    ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;
----------------------------------------------------------------------
Command executed successfully.

Statement 2/2
WITH delivery_flags AS (
    SELECT
        order_id,
        (
            order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE
        ) AS is_late_calc
    FROM olist_orders
)
UPDATE olist_orders AS o
SET is_late = f.is_late_calc
FROM delivery_flags AS f
WHERE f.order_id = o.order_id
RETURNING o.order_id, o.customer_id, o.is_late;
----------------------------------------------------------------------


,order_id,customer_id,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,False
...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,False
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,False
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,False
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,False



Finished successfully: 2 statement(s) executed.


**Пояснення.** `UPDATE ... FROM` заново рахує `is_late` для кожного замовлення з поточних дат доставки. Тому повторний запуск дає ті самі значення, а не дописує `TRUE` поверх старих. `RETURNING` показує, які рядки оновилися.

In [16]:
execute_sql("""
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at;
""")


Statement 1/1
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at;
----------------------------------------------------------------------


,alert_id,seller_id,alert_type,created_at



Finished successfully: 1 statement(s) executed.


**Пояснення.** Запит видаляє alerts, старші за 30 днів, і через `RETURNING` показує, що саме видалено. Тут результат порожній, і це нормально: єдиний alert створено кількома клітинками вище, йому кілька секунд. Повторний запуск з тією самою умовою нічого не змінить.

## 3. Сегментація покупців

Спершу будую `olist_customer_dim`, де по одному рядку на `customer_unique_id`. Потім `customer_segments` з identity PK, FK на dim-таблицю, `UNIQUE`, `CHECK`, `NOT NULL` і `DEFAULT`.

In [17]:
execute_sql("""
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT REFERENCES olist_customers(customer_id),
    customer_state     CHAR(2),
    customer_city      TEXT
);

INSERT INTO olist_customer_dim (
    customer_unique_id,
    latest_customer_id,
    customer_state,
    customer_city
)
SELECT DISTINCT ON (c.customer_unique_id)
    c.customer_unique_id,
    c.customer_id AS latest_customer_id,
    c.customer_state,
    c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
ORDER BY
    c.customer_unique_id,
    o.order_purchase_timestamp DESC NULLS LAST,
    c.customer_id;
""")


Statement 1/4
DROP TABLE IF EXISTS customer_segments CASCADE;
----------------------------------------------------------------------
Command executed successfully.

Statement 2/4
DROP TABLE IF EXISTS olist_customer_dim CASCADE;
----------------------------------------------------------------------
Command executed successfully.

Statement 3/4
CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT REFERENCES olist_customers(customer_id),
    customer_state     CHAR(2),
    customer_city      TEXT
);
----------------------------------------------------------------------
Command executed successfully.

Statement 4/4
INSERT INTO olist_customer_dim (
    customer_unique_id,
    latest_customer_id,
    customer_state,
    customer_city
)
SELECT DISTINCT ON (c.customer_unique_id)
    c.customer_unique_id,
    c.customer_id AS latest_customer_id,
    c.customer_state,
    c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o
    

In [18]:
execute_sql("""
CREATE TABLE customer_segments (
    segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    customer_unique_id  TEXT NOT NULL,
    segment_name        TEXT NOT NULL
        CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),

    rfm_score           SMALLINT NOT NULL
        CHECK (rfm_score BETWEEN 1 AND 5),

    recency_days        INTEGER
        CHECK (recency_days IS NULL OR recency_days >= 0),

    monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0
        CHECK (monetary_value >= 0),

    n_orders            INTEGER NOT NULL DEFAULT 0
        CHECK (n_orders >= 0),

    assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;
""")


Statement 1/2
CREATE TABLE customer_segments (
    segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    customer_unique_id  TEXT NOT NULL,
    segment_name        TEXT NOT NULL
        CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),

    rfm_score           SMALLINT NOT NULL
        CHECK (rfm_score BETWEEN 1 AND 5),

    recency_days        INTEGER
        CHECK (recency_days IS NULL OR recency_days >= 0),

    monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0
        CHECK (monetary_value >= 0),

    n_orders            INTEGER NOT NULL DEFAULT 0
        CHECK (n_orders >= 0),

    assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    UNIQUE (customer_unique_id)
);
----------------------------------------------------------------------
Command executed successfully.

Statement 2/2
ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(cu

In [19]:
execute_sql("""
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp
),
customer_stats AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
        MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c
        ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot
        ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT
        cs.customer_unique_id,
        cs.n_orders,
        cs.monetary_value,
        CASE
            WHEN cs.last_order_date IS NULL THEN NULL
            ELSE (rd.as_of_date - cs.last_order_date)::INTEGER
        END AS recency_days,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 'VIP'
            WHEN cs.last_order_date IS NOT NULL
                 AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                 AND cs.n_orders = 1 THEN 'new'
            WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
            ELSE 'inactive'
        END AS segment_name,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 5
            WHEN cs.monetary_value >= 500  THEN 4
            WHEN cs.n_orders >= 2          THEN 3
            WHEN cs.n_orders = 1           THEN 2
            ELSE 1
        END::SMALLINT AS rfm_score
    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
)
INSERT INTO customer_segments (
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
)
SELECT
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
FROM segmented
ON CONFLICT (customer_unique_id) DO UPDATE
SET segment_name   = EXCLUDED.segment_name,
    rfm_score      = EXCLUDED.rfm_score,
    recency_days   = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders       = EXCLUDED.n_orders,
    assigned_at    = NOW()
RETURNING
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders;
""")


Statement 1/1
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp
),
customer_stats AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
        MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c
        ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot
        ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id

,customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,regular,2,161,141.90,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,inactive,2,164,27.19,1
2,0000f46a3911fa3c0805444483337064,inactive,2,587,86.22,1
3,0000f6ccb0745a6a4b88665a16c9f078,inactive,2,371,43.62,1
4,0004aac84e0df4da2b147fca70cf8255,regular,2,338,196.89,1
...,...,...,...,...,...,...
96091,fffcf5a5ff07b0908bd4e2dbc735a684,VIP,5,497,2067.42,1
96092,fffea47cd6d3cc0a88bd621562a9d061,inactive,2,312,84.58,1
96093,ffff371b4d645b6ecea244b27531430a,regular,2,618,112.46,1
96094,ffff5962728ec6157033ef9805bacc48,regular,2,169,133.69,1



Finished successfully: 1 statement(s) executed.


In [20]:
execute_sql("""
SELECT
    conname,
    contype,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname;
""")


Statement 1/1
SELECT
    conname,
    contype,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname;
----------------------------------------------------------------------


,conname,contype,definition
0,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
1,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
2,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
3,customer_segments_pkey,p,PRIMARY KEY (segment_id)
4,customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_day...
5,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
6,customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text,..."
7,fk_customer_segments_customer_unique,f,FOREIGN KEY (customer_unique_id) REFERENCES ol...



Finished successfully: 1 statement(s) executed.


**Пояснення.**
- Сегментую за `customer_unique_id`, бо це і є реальна людина. `customer_id` в Olist видається на кожне замовлення, тож при групуванні за ним майже в усіх було б `n_orders = 1`.
- FK має на щось посилатися, а в `olist_customers` поле `customer_unique_id` не унікальне. Тому й потрібна `olist_customer_dim`, де воно є PK.
- Дату відліку беру як день після останнього замовлення в датасеті. Дані закінчуються 2018 роком, тож із `CURRENT_DATE` recency в усіх був би кілька років і змінювався б щодня.
- `ON CONFLICT (customer_unique_id) DO UPDATE` при повторному запуску оновлює сегмент наявного покупця замість помилки на `UNIQUE`.

У виводі `pg_constraint` видно PK, FK, `UNIQUE` і всі `CHECK`. `NOT NULL` і `DEFAULT` туди не потрапляють, вони задані прямо в DDL.

## 4. JOIN-запити

П’ять запитів: `INNER`, `LEFT` + `COALESCE`, `FULL OUTER`, `SELF` і anti-join. Питання й SQL придумував сам, приклади із завдання не копіював.

### JOIN 1 — INNER: категорії в замовленні

**Business-question:** скільки різних категорій товарів купили в межах одного `customer_id` і які саме? Тут `customer_id` фактично означає одне замовлення, а не всю історію покупця.

In [21]:
execute_sql("""
SELECT DISTINCT oc.customer_id, COUNT(DISTINCT op.product_category_name) AS count, CASE JSONB_AGG(DISTINCT op.product_category_name)::TEXT WHEN '[null]' THEN '[]' ELSE JSONB_AGG(DISTINCT op.product_category_name)::TEXT END AS list
FROM olist_customers AS oc
JOIN olist_orders AS oo
    ON oc.customer_id = oo.customer_id
JOIN olist_order_items AS ooi
    ON oo.order_id = ooi.order_id
JOIN olist_products AS op
    ON ooi.product_id = op.product_id
GROUP BY oc.customer_id
ORDER BY count DESC;
""")


Statement 1/1
SELECT DISTINCT oc.customer_id, COUNT(DISTINCT op.product_category_name) AS count, CASE JSONB_AGG(DISTINCT op.product_category_name)::TEXT WHEN '[null]' THEN '[]' ELSE JSONB_AGG(DISTINCT op.product_category_name)::TEXT END AS list
FROM olist_customers AS oc
JOIN olist_orders AS oo
    ON oc.customer_id = oo.customer_id
JOIN olist_order_items AS ooi
    ON oo.order_id = ooi.order_id
JOIN olist_products AS op
    ON ooi.product_id = op.product_id
GROUP BY oc.customer_id
ORDER BY count DESC;
----------------------------------------------------------------------


,customer_id,count,list
0,0aad2e31b3c119c26acb8a47768cd00a,3,"[""bebes"", ""cama_mesa_banho"", ""cool_stuff""]"
1,21de634855f444443723e6c8b224ae69,3,"[""cama_mesa_banho"", ""moveis_decoracao"", ""movei..."
2,2306cc5d9784d518dce0f1d7649b1d61,3,"[""cama_mesa_banho"", ""market_place"", ""perfumaria""]"
3,33e66871536941873f7c041f120012be,3,"[""casa_construcao"", ""ferramentas_jardim"", ""mov..."
4,3d338ae6cd95dcb8c0c14d9af90055f3,3,"[""automotivo"", ""cama_mesa_banho"", ""cool_stuff""]"
...,...,...,...
98661,ff88c3fe67d72e79283259628c0731e1,0,[]
98662,ff96b3b15e8f7a878bede85345e33f0c,0,[]
98663,ffa78a5bd171f26facc78c0409e86d3f,0,[]
98664,ffbd6fd1ff815c8392bb0f6e2fdeef75,0,[]



Finished successfully: 1 statement(s) executed.


**Interpretation.** Вийшло 98 666 `customer_id`, у яких є позиції замовлення. Максимум — 3 різні категорії. Через `INNER JOIN` замовлення без позицій сюди не потрапили. Нуль у `count` означає, що в товару не вказана категорія. Повторні покупки однієї людини тут не склеюються, для цього далі використовую `customer_unique_id`.

### JOIN 2 — LEFT + COALESCE: ознаки покупця

**Business-question:** на яку суму (за цінами товарів) купив кожен реальний покупець (`customer_unique_id`) і скільки позицій він узяв?

In [22]:
execute_sql("""
SELECT oc.customer_unique_id, COALESCE(SUM(ooi.price),0) AS total_price, COUNT(ooi.product_id) as total_items
FROM olist_customers AS oc
LEFT JOIN olist_orders AS oo
    ON oc.customer_id = oo.customer_id
LEFT JOIN olist_order_items AS ooi
    ON oo.order_id = ooi.order_id
GROUP BY oc.customer_unique_id
ORDER BY total_price DESC;
""")


Statement 1/1
SELECT oc.customer_unique_id, COALESCE(SUM(ooi.price),0) AS total_price, COUNT(ooi.product_id) as total_items
FROM olist_customers AS oc
LEFT JOIN olist_orders AS oo
    ON oc.customer_id = oo.customer_id
LEFT JOIN olist_order_items AS ooi
    ON oo.order_id = ooi.order_id
GROUP BY oc.customer_unique_id
ORDER BY total_price DESC;
----------------------------------------------------------------------


,customer_unique_id,total_price,total_items
0,0a0a92112bd4c708ca5fde585afaa872,13440.00,8
1,da122df9eeddfedc1dc1f5349a1a690c,7388.00,2
2,763c8b1c9c68a0229c42c9fc6f662b93,7160.00,4
3,dc4802a71eae9be1dd28f5d788ceb526,6735.00,1
4,459bef486812aa25204be022145caa62,6729.00,1
...,...,...,...
96091,e50756a1f601b707158af7993609f51e,0,0
96092,e9bd60258117342273c84198e732b955,0,0
96093,e9afee56bf88242799eaf0f0c2853c1b,0,0
96094,e72c31d513b0da68b1ea23dd4375257b,0,0



Finished successfully: 1 statement(s) executed.


**Interpretation.** 96 096 рядків, по одному на покупця. Найбільша сума — 13 440,00 за 8 позицій. `LEFT JOIN` залишає й тих, у кого немає жодної позиції, а `COALESCE` ставить їм 0 замість `NULL`. Відгуки тут не приєднуються, тому множитися нічому.

### JOIN 3 — FULL OUTER: два типи ризику в продавців

**Business-question:** скільки продавців мають запізнілі доставки, скільки мають середній рейтинг нижче 3 і скільки потрапляють в обидві групи? Ці два списки незалежні, тому беру `FULL OUTER JOIN`, щоб не загубити продавців, які є тільки в одному з них.

In [23]:
execute_sql("""
WITH late_sellers AS (
    SELECT DISTINCT oi.seller_id
    FROM olist_order_items AS oi
    JOIN olist_orders AS o
        ON o.order_id = oi.order_id
    WHERE o.is_late = TRUE
),
low_rating_sellers AS (
    SELECT seller_id
    FROM seller_score
    WHERE avg_rating < 3
),
compared AS (
    SELECT
        COALESCE(l.seller_id, r.seller_id) AS seller_id,
        CASE
            WHEN l.seller_id IS NOT NULL AND r.seller_id IS NOT NULL THEN 'both'
            WHEN l.seller_id IS NOT NULL THEN 'late_only'
            ELSE 'low_rating_only'
        END AS risk_group
    FROM late_sellers AS l
    FULL OUTER JOIN low_rating_sellers AS r
        ON l.seller_id = r.seller_id
)
SELECT risk_group, COUNT(*) AS n_sellers
FROM compared
GROUP BY risk_group
ORDER BY risk_group;
""")


Statement 1/1
WITH late_sellers AS (
    SELECT DISTINCT oi.seller_id
    FROM olist_order_items AS oi
    JOIN olist_orders AS o
        ON o.order_id = oi.order_id
    WHERE o.is_late = TRUE
),
low_rating_sellers AS (
    SELECT seller_id
    FROM seller_score
    WHERE avg_rating < 3
),
compared AS (
    SELECT
        COALESCE(l.seller_id, r.seller_id) AS seller_id,
        CASE
            WHEN l.seller_id IS NOT NULL AND r.seller_id IS NOT NULL THEN 'both'
            WHEN l.seller_id IS NOT NULL THEN 'late_only'
            ELSE 'low_rating_only'
        END AS risk_group
    FROM late_sellers AS l
    FULL OUTER JOIN low_rating_sellers AS r
        ON l.seller_id = r.seller_id
)
SELECT risk_group, COUNT(*) AS n_sellers
FROM compared
GROUP BY risk_group
ORDER BY risk_group;
----------------------------------------------------------------------


,risk_group,n_sellers
0,both,86
1,late_only,1188
2,low_rating_only,219



Finished successfully: 1 statement(s) executed.


**Interpretation.** 86 продавців мають обидві проблеми, 1 188 — тільки запізнення, 219 — тільки низький рейтинг. Звичайний `JOIN` залишив би лише групу `both`. Запізнення трапляються значно частіше за поганий рейтинг, і здебільшого це різні продавці. Рейтинг має те саме обмеження, що й у `seller_score`: відгук стосується всього замовлення.

### JOIN 4 — SELF: продавці з однаковим ZIP-префіксом

**Business-question:** які пари продавців мають однаковий поштовий префікс, тобто працюють фактично поруч, і їх можна порівнювати між собою?

In [24]:
execute_sql("""
SELECT
    s1.seller_id AS seller_1,
    s2.seller_id AS seller_2,
    s1.seller_zip_code_prefix,
    s1.seller_city,
    s1.seller_state
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_zip_code_prefix = s2.seller_zip_code_prefix
    AND s1.seller_id < s2.seller_id
WHERE s1.seller_zip_code_prefix IS NOT NULL
ORDER BY s1.seller_zip_code_prefix
LIMIT 100;
""")


Statement 1/1
SELECT
    s1.seller_id AS seller_1,
    s2.seller_id AS seller_2,
    s1.seller_zip_code_prefix,
    s1.seller_city,
    s1.seller_state
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_zip_code_prefix = s2.seller_zip_code_prefix
    AND s1.seller_id < s2.seller_id
WHERE s1.seller_zip_code_prefix IS NOT NULL
ORDER BY s1.seller_zip_code_prefix
LIMIT 100;
----------------------------------------------------------------------


,seller_1,seller_2,seller_zip_code_prefix,seller_city,seller_state
0,dd55f1bb788714a40e7954c3be6df745,e0487761face83d64fcada2408959a36,1021,sao paulo,SP
1,82921991ff5b557b045605b8bbf08d49,f049a72cf58fd31b11f8919cade515e7,1023,sao paulo,SP
2,14a08204d03bb6b6bde8029f801ae0eb,5d378b73ab7dd6f0418d743e5dcb0bd1,1026,sao paulo,SP
3,14a08204d03bb6b6bde8029f801ae0eb,6da1992f915d77be95d7fa48b36904af,1026,sao paulo,SP
4,14a08204d03bb6b6bde8029f801ae0eb,630008e0f062605a415d694489f6f82e,1026,sao paulo,SP
...,...,...,...,...,...
95,b2a6d334e2833acea353624840e25a0e,b87a7ea0c30fad2dd6781b74a0064859,4133,sao paulo,SP
96,11fb6f6d341adbe19e81733701704635,b87a7ea0c30fad2dd6781b74a0064859,4133,sao paulo,SP
97,11fb6f6d341adbe19e81733701704635,b2a6d334e2833acea353624840e25a0e,4133,sao paulo,SP
98,253f95b3a0e6983ae18aed49fbc2b845,818d28f936e791926d33bcb6ee94ef79,4141,sao paulo,SP



Finished successfully: 1 statement(s) executed.


**Interpretation.** Показав перші 100 пар. Умова `s1.seller_id < s2.seller_id` прибирає пари продавця із самим собою і дзеркальні дублікати (A–B і B–A). Однаковий префікс означає близьке розташування, але не ту саму адресу.

### JOIN 5 — anti-join: товари, яких ніхто не купував

**Business-question:** чи є в каталозі товари, які не трапляються в жодній позиції замовлення? Anti-join зроблено через `LEFT JOIN ... IS NULL`.

In [25]:
execute_sql("""
SELECT
    p.product_id,
    p.product_category_name
FROM olist_products AS p
LEFT JOIN olist_order_items AS oi
    ON p.product_id = oi.product_id
WHERE oi.product_id IS NULL;
""")


Statement 1/1
SELECT
    p.product_id,
    p.product_category_name
FROM olist_products AS p
LEFT JOIN olist_order_items AS oi
    ON p.product_id = oi.product_id
WHERE oi.product_id IS NULL;
----------------------------------------------------------------------


,product_id,product_category_name



Finished successfully: 1 statement(s) executed.


**Interpretation.** Таких товарів немає: кожен товар з `olist_products` хоча б раз є в `order_items`. Схоже, у датасет потрапили тільки товари, які продавалися. Про реальний асортимент чи залишки на складі це нічого не говорить.

## 5. GROUP BY, HAVING, агрегати

Три метрики з групуванням за категорією, покупцем і замовленням. `HAVING` є в першому й третьому запиті, `JSONB_AGG` — у другому.

### Метрика 1 — середня оцінка категорії

**Business-question:** у яких категорій середня оцінка замовлень вища за 3? Щоб кілька позицій однієї категорії в замовленні не рахувалися кілька разів, спочатку беру унікальні пари `order_id × category`, а відгуки зводжу до рівня `order_id`.

In [26]:
execute_sql("""
WITH order_categories AS (
    SELECT DISTINCT oi.order_id, p.product_category_name
    FROM olist_order_items AS oi
    JOIN olist_products AS p
        ON p.product_id = oi.product_id
),
review_per_order AS (
    SELECT order_id, AVG(review_score) AS order_review_score
    FROM olist_order_reviews
    WHERE review_score IS NOT NULL
    GROUP BY order_id
)
SELECT
    oc.product_category_name,
    AVG(rpo.order_review_score) AS avg_order_review_score
FROM order_categories AS oc
JOIN review_per_order AS rpo
    ON rpo.order_id = oc.order_id
GROUP BY oc.product_category_name
HAVING AVG(rpo.order_review_score) > 3
ORDER BY avg_order_review_score DESC;
""")


Statement 1/1
WITH order_categories AS (
    SELECT DISTINCT oi.order_id, p.product_category_name
    FROM olist_order_items AS oi
    JOIN olist_products AS p
        ON p.product_id = oi.product_id
),
review_per_order AS (
    SELECT order_id, AVG(review_score) AS order_review_score
    FROM olist_order_reviews
    WHERE review_score IS NOT NULL
    GROUP BY order_id
)
SELECT
    oc.product_category_name,
    AVG(rpo.order_review_score) AS avg_order_review_score
FROM order_categories AS oc
JOIN review_per_order AS rpo
    ON rpo.order_id = oc.order_id
GROUP BY oc.product_category_name
HAVING AVG(rpo.order_review_score) > 3
ORDER BY avg_order_review_score DESC;
----------------------------------------------------------------------


,product_category_name,avg_order_review_score
0,cds_dvds_musicais,4.6666666666666667
1,fashion_roupa_infanto_juvenil,4.50000000000000000000
2,livros_interesse_geral,4.46259842519685039370
3,construcao_ferramentas_ferramentas,4.42553191489361702128
4,livros_tecnicos,4.40077821011673151751
...,...,...
68,fashion_roupa_feminina,3.73076923076923076923
69,fashion_roupa_masculina,3.70270270270270270270
70,moveis_escritorio,3.61638954869358669834
71,portateis_cozinha_e_preparadores_de_alimentos,3.42857142857142857143



Finished successfully: 1 statement(s) executed.


**Interpretation.** Поріг пройшли 73 категорії, найвища оцінка в `cds_dvds_musicais` (~4,67). Кожне замовлення в межах категорії враховане один раз. Але якщо в замовленні кілька категорій, його оцінка йде кожній із них, тож це наближення.

### Метрика 2 — скільки категорій купує покупець

**Business-question:** скільки різних категорій купив кожен реальний покупець і які саме? Від JOIN 1 відрізняється тим, що групую за `customer_unique_id`. Список збирає `JSONB_AGG`.

In [27]:
execute_sql("""
SELECT DISTINCT oc.customer_unique_id, COUNT(DISTINCT op.product_category_name) AS count, CASE JSONB_AGG(DISTINCT op.product_category_name)::TEXT WHEN '[null]' THEN '[]' ELSE JSONB_AGG(DISTINCT op.product_category_name)::TEXT END AS list
FROM olist_customers AS oc
JOIN olist_orders AS oo
    ON oc.customer_id = oo.customer_id
JOIN olist_order_items AS ooi
    ON oo.order_id = ooi.order_id
JOIN olist_products AS op
    ON ooi.product_id = op.product_id
GROUP BY oc.customer_unique_id
ORDER BY count DESC;
""")


Statement 1/1
SELECT DISTINCT oc.customer_unique_id, COUNT(DISTINCT op.product_category_name) AS count, CASE JSONB_AGG(DISTINCT op.product_category_name)::TEXT WHEN '[null]' THEN '[]' ELSE JSONB_AGG(DISTINCT op.product_category_name)::TEXT END AS list
FROM olist_customers AS oc
JOIN olist_orders AS oo
    ON oc.customer_id = oo.customer_id
JOIN olist_order_items AS ooi
    ON oo.order_id = ooi.order_id
JOIN olist_products AS op
    ON ooi.product_id = op.product_id
GROUP BY oc.customer_unique_id
ORDER BY count DESC;
----------------------------------------------------------------------


,customer_unique_id,count,list
0,1b6c7548a2a1f9037c1fd3ddfed95f33,5,"[""beleza_saude"", ""moveis_decoracao"", ""papelari..."
1,35ecdf6858edc6427223b64804cf028e,5,"[""alimentos_bebidas"", ""beleza_saude"", ""esporte..."
2,3e43e6105506432c953e165fb2acf44c,5,"[""cama_mesa_banho"", ""casa_construcao"", ""inform..."
3,47c1a3033b8b77b3ab6e109eb4d5fdf3,5,"[""automotivo"", ""beleza_saude"", ""cama_mesa_banh..."
4,74cb1ad7e6d5674325c1f99b5ea30d82,5,"[""climatizacao"", ""construcao_ferramentas_const..."
...,...,...,...
95415,ffaa5b5559566bb4dbf71d554310e760,0,[]
95416,ffb2ae2aa0951d0885d7cd9a218b0c46,0,[]
95417,ffbd1090fa1840d6cf9106fb238089e9,0,[]
95418,ffdf72e25c6f0503bb567d0a8cf1c159,0,[]



Finished successfully: 1 statement(s) executed.


**Interpretation.** 95 420 покупців із позиціями, максимум — 5 різних категорій. Тут уже видно повторні покупки: у JOIN 1 максимум був 3, бо замовлення однієї людини там рахувалися окремо. Покупці без позицій через `INNER JOIN` не потрапили.

### Метрика 3 — дорогі скасовані замовлення

**Business-question:** які скасовані замовлення мали суму товарів понад 1000? Доставку не враховую.

In [28]:
execute_sql("""
SELECT
    oo.order_id,
    oo.customer_id,
    SUM(ooi.price) AS total_price
FROM olist_customers AS oc
LEFT JOIN olist_orders AS oo
    ON oc.customer_id = oo.customer_id
LEFT JOIN olist_order_items AS ooi
    ON oo.order_id = ooi.order_id
WHERE oo.order_status = 'canceled'
GROUP BY oo.order_id
HAVING SUM(ooi.price) > 1000
ORDER BY total_price DESC;
""")


Statement 1/1
SELECT
    oo.order_id,
    oo.customer_id,
    SUM(ooi.price) AS total_price
FROM olist_customers AS oc
LEFT JOIN olist_orders AS oo
    ON oc.customer_id = oo.customer_id
LEFT JOIN olist_order_items AS ooi
    ON oo.order_id = ooi.order_id
WHERE oo.order_status = 'canceled'
GROUP BY oo.order_id
HAVING SUM(ooi.price) > 1000
ORDER BY total_price DESC;
----------------------------------------------------------------------


,order_id,customer_id,total_price
0,b4c4b76c642808cbe472a32b86cddc95,e0a2412720e9ea4f26c1ac985f6a7358,4599.90
1,83e6338b5cf25dcf222551cb8da8d0d6,3690e975641f01bd07ad635f03dbb894,2649.00
2,078f6a01964ee122ef20881df839af31,2b7ae7a532ff538ce88f6bfdd2cb564c,2350.00
3,f7b13a5afd48ca5bcf589a25d5377b00,d791834a0555c0062e98d735083f4373,2299.95
4,98464a5096e400810b94bb93f38924c9,24301c6ce48874a51221d4e5584a7885,2299.95
5,e23eaa3bc8275c392145e736dbdea275,50bb775cd30a21ae108d4866abb6d9df,1999.99
6,c87de2d1aece481642a24bae4b964b07,e91409937a0fd26a57310548858d1762,1854.00
7,9f4f60c24b20845efd8d6b863b571d20,349f53f02e5fd1cf1a2a9c7faf422363,1800.00
8,a9c8ac0c26c178f0ad33618f96225a01,727ab4449f530b9af13f40fa8e528bff,1795.00
9,73863aca434e1d117e45109bea6f1937,f3f7e581618408da1e8d5ebe236082b3,1669.87



Finished successfully: 1 statement(s) executed.


**Interpretation.** Таких замовлень 17, найдорожче — на 4 599,90. Тут добре видно різницю між `WHERE` і `HAVING`. `WHERE order_status = 'canceled'` відкидає рядки ще до групування, а `HAVING SUM(ooi.price) > 1000` відкидає вже готові групи (замовлення) за порахованою сумою. Умову на суму в `WHERE` не написати, бо на той момент суми ще не існує.

## 6. Set operation і window function

**Business-question для `EXCEPT`:** чи є штати, де є продавці, але немає жодного покупця? Узяв `EXCEPT`, бо потрібне саме те, що є в першому наборі й відсутнє в другому. `UNION` чи `INTERSECT` дали б інше питання.

In [29]:
execute_sql("""
SELECT seller_state AS state FROM olist_sellers

EXCEPT

SELECT customer_state AS state FROM olist_customers

ORDER BY state;
""")


Statement 1/1
SELECT seller_state AS state FROM olist_sellers

EXCEPT

SELECT customer_state AS state FROM olist_customers

ORDER BY state;
----------------------------------------------------------------------


,state



Finished successfully: 1 statement(s) executed.


**Interpretation.** Порожньо: у кожному штаті з продавцями є й покупці. `EXCEPT` сам прибирає дублікати, тому `DISTINCT` не знадобився. Порожній результат тут теж є відповіддю на питання.

### Window function: найдорожчі позиції продавця

**Business-question:** які позиції в кожного продавця найдорожчі? Використовую `RANK()`, щоб однакові ціни отримували однаковий ранг.

In [30]:
execute_sql("""
SELECT
    seller_id,
    product_id,
    price,
    RANK() OVER (
        PARTITION BY seller_id
        ORDER BY price DESC
    ) AS price_rank
FROM olist_order_items
ORDER BY seller_id ASC, price_rank ASC;
""")


Statement 1/1
SELECT
    seller_id,
    product_id,
    price,
    RANK() OVER (
        PARTITION BY seller_id
        ORDER BY price DESC
    ) AS price_rank
FROM olist_order_items
ORDER BY seller_id ASC, price_rank ASC;
----------------------------------------------------------------------


,seller_id,product_id,price,price_rank
0,0015a82c2db000af6aaaf3ae2ecb0532,a2ff5a97bf95719e38ea2e3b4105bce8,895.00,1
1,0015a82c2db000af6aaaf3ae2ecb0532,a2ff5a97bf95719e38ea2e3b4105bce8,895.00,1
2,0015a82c2db000af6aaaf3ae2ecb0532,a2ff5a97bf95719e38ea2e3b4105bce8,895.00,1
3,001cca7ae9ae17fb1caed9dfb1094831,e251ebd2858be1aa7d9b2087a6992580,199.00,1
4,001cca7ae9ae17fb1caed9dfb1094831,98a8c2fa16d7239c606640f5555768e4,169.00,2
...,...,...,...,...
112645,ffff564a4f9085cd26170f4732393726,dbd024d4182504993ad1e3cd2ee9d9e9,29.40,16
112646,ffff564a4f9085cd26170f4732393726,dbd024d4182504993ad1e3cd2ee9d9e9,29.40,16
112647,ffff564a4f9085cd26170f4732393726,c7d75b3bafed2b1d9475c4876b1c46b8,15.65,18
112648,ffff564a4f9085cd26170f4732393726,c4b925e40f11289063a854c47aaef129,11.50,19



Finished successfully: 1 statement(s) executed.


**Interpretation.** Ранг пораховано для всіх 112 650 позицій. Однакові ціни в продавця мають однаковий ранг, а наступний номер після них пропускається (1, 1, 3...). Ранжуються продажі, а не унікальні товари, тому один `product_id` може йти кілька разів поспіль.

## Reflection

Найбільше проблем було із SELF JOIN. Перша версія порівнювала продавців надто широко, пар вийшло майже як у декартовому добутку, і запит вичерпав усі 50 ГБ оперативки. Довелося обмежити пари однаковим поштовим префіксом і додати `s1.seller_id < s2.seller_id`, щоб прибрати самопари й дзеркальні дублікати. Висновок для себе: умову join-у треба продумувати до запуску, а не після.

Найдовше думав над business-question для FULL OUTER JOIN. Потрібні два незалежні набори, де і «тільки зліва», і «тільки справа» щось означають. Зупинився на продавцях із запізненнями та з рейтингом нижче 3: вийшло 86 з обома проблемами, 1188 тільки із запізненнями і 219 тільки з низьким рейтингом. Так видно, де проблема в логістиці, а де в сервісі.

Для production ETL найкориснішим вважаю `ON CONFLICT`: `seller_score` і `customer_segments` можна перераховувати скільки завгодно без дублікатів. Window functions в ML-пайплайні я б використав, щоб брати останнє замовлення покупця, ранжувати товари продавця й рахувати інтервали між покупками. Головне при цьому — не захопити в ознаки майбутні події.

Щодо якості даних: 768 замовлень без відгуку, 610 товарів без категорії, 58 247 відгуків без тексту, а `review_id` подекуди повторюється. Ознаки покупця рахую за `customer_unique_id`, бо `customer_id` видається на кожне замовлення і розриває історію повторних покупок. Перед `seller_score` зводжу дані до `seller_id × order_id`, а відгуки до `order_id`, інакше одна оцінка отримала б більшу вагу через кілька товарів. Якщо в замовленні кілька продавців, оцінка все одно дістається кожному, і це вже обмеження самих даних.